# GSS Polarization Analysis

This notebook calculates polarization scores for GSS questions across two categories:
- **Public Issues**: Policy and political opinion questions
- **Private Life**: Personal lifestyle and values questions

Polarization is measured as the normalized difference in mean responses between Democrats and Republicans.

In [27]:
import sys; sys.path.insert(0, "..")
import pandas as pd
import numpy as np
from gss_topic_mappings import PUBLIC_ISSUES_TOPICS, PRIVATE_LIFE_TOPICS

print(f"PUBLIC_ISSUES_TOPICS: {len(PUBLIC_ISSUES_TOPICS)} groups")
print(f"PRIVATE_LIFE_TOPICS: {len(PRIVATE_LIFE_TOPICS)} groups")

PUBLIC_ISSUES_TOPICS: 60 groups
PRIVATE_LIFE_TOPICS: 48 groups


In [28]:
from gss_categorized_variables import PUBLIC_ISSUES, PRIVATE_LIFE

In [29]:
# Flatten PUBLIC_ISSUES_TOPICS and PRIVATE_LIFE_TOPICS into sets of base variable names
def flatten_topics(topic_dict):
    flat = set()
    for vars_list in topic_dict.values():
        for v in vars_list:
            flat.add(v)
    return flat

public_flat = flatten_topics(PUBLIC_ISSUES_TOPICS)
private_flat = flatten_topics(PRIVATE_LIFE_TOPICS)

print(f"public_flat: {len(public_flat)} variables")
print(f"private_flat: {len(private_flat)} variables")

public_flat: 172 variables
private_flat: 112 variables


In [30]:
# Compare public_flat with PUBLIC_ISSUES

public_issues_set = set(PUBLIC_ISSUES)

# Variables in PUBLIC_ISSUES but not in public_flat
in_public_issues_not_flat = public_issues_set - public_flat
print(f"In PUBLIC_ISSUES but NOT in public_flat: {len(in_public_issues_not_flat)}")
if in_public_issues_not_flat:
    print(f"  {sorted(in_public_issues_not_flat)}")

# Variables in public_flat but not in PUBLIC_ISSUES
in_flat_not_public_issues = public_flat - public_issues_set
print(f"\nIn public_flat but NOT in PUBLIC_ISSUES: {len(in_flat_not_public_issues)}")
if in_flat_not_public_issues:
    print(f"  {sorted(in_flat_not_public_issues)}")

# Variables in both
in_both = public_flat & public_issues_set
print(f"\nIn both: {len(in_both)}")

In PUBLIC_ISSUES but NOT in public_flat: 0

In public_flat but NOT in PUBLIC_ISSUES: 0

In both: 172


In [31]:
# Load the survey data
df = pd.read_csv('../../data/gss/gss_2021_2024.csv')
print(f"Loaded {len(df)} respondents")
print(f"Years: {df['year'].unique()}")

# Filter for valid party ID (0-6 scale)
df = df[df['partyid'].between(0, 6)]
print(f"After filtering for valid partyid: {len(df)} respondents")
print(f"\nParty ID distribution:\n{df['partyid'].value_counts().sort_index()}")

Loaded 11066 respondents
Years: [2021 2022 2024]
After filtering for valid partyid: 11066 respondents

Party ID distribution:
partyid
0.0    2064
1.0    1506
2.0    1290
3.0    2617
4.0     967
5.0    1179
6.0    1443
Name: count, dtype: int64


In [32]:
# Merge duplicate variables (base and 'y' suffix are same question on different ballots)
DUPLICATE_PAIRS = [
    ('natspac', 'natspacy'),
    ('natenvir', 'natenviy'),
    ('natheal', 'nathealy'),
    ('natcity', 'natcityy'),
    ('natcrime', 'natcrimy'),
    ('natdrug', 'natdrugy'),
    ('nateduc', 'nateducy'),
    ('natrace', 'natracey'),
    ('natarms', 'natarmsy'),
    ('nataid', 'nataidy'),
    ('natfare', 'natfarey'),
    ('letdie1', 'letdie1y'),
    ('spkath', 'spkathy'),
    ('libath', 'libathy'),
    ('spkcom', 'spkcomy'),
    ('spkhomo', 'spkhomoy'),
    ('spkmil', 'spkmily'),
    ('spkmslm', 'spkmslmy'),
    ('spkrac', 'spkracy'),
    ('helpful', 'helpfulnv'),
    ('helpful', 'helpfulv'),
]

for base, alt in DUPLICATE_PAIRS:
    if base in df.columns and alt in df.columns:
        # Combine: use base if available, otherwise use alt
        df[base] = df[base].combine_first(df[alt])
        df = df.drop(columns=[alt])
        print(f"  Merged {alt} into {base}")
    elif alt in df.columns and base not in df.columns:
        # Rename alt to base
        df = df.rename(columns={alt: base})
        print(f"  Renamed {alt} to {base}")

print(f"\nColumns after merging: {len(df.columns)}")

  Merged natspacy into natspac
  Merged natenviy into natenvir
  Merged nathealy into natheal
  Merged natcityy into natcity
  Merged natcrimy into natcrime
  Merged natdrugy into natdrug
  Merged nateducy into nateduc
  Merged natracey into natrace
  Merged natarmsy into natarms
  Merged nataidy into nataid
  Merged natfarey into natfare
  Merged letdie1y into letdie1
  Merged spkathy into spkath
  Merged libathy into libath
  Merged spkcomy into spkcom
  Merged spkhomoy into spkhomo
  Merged spkmily into spkmil
  Merged spkmslmy into spkmslm
  Merged spkracy into spkrac
  Merged helpfulnv into helpful
  Merged helpfulv into helpful

Columns after merging: 1750


In [33]:
# Polarization calculation functions

def create_party_groups(df, partyid_col='partyid'):
    """
    Create party group labels from partyid (0-6 scale).
    0-2: Democrat (Strong Dem, Not Strong Dem, Ind Near Dem)
    3: Independent
    4-6: Republican (Ind Near Rep, Not Strong Rep, Strong Rep)
    """
    df = df.copy()
    conditions = [
        df[partyid_col].between(0, 2),
        df[partyid_col] == 3,
        df[partyid_col].between(4, 6)
    ]
    choices = ['Democrat', 'Independent', 'Republican']
    df['party_group'] = np.select(conditions, choices, default='Other')
    return df

def calculate_polarization(df, question_col, partyid_col='partyid', min_responses=30):
    """
    Calculate normalized partisan gap for a single question.
    
    Returns dict with:
    - polarization: normalized |mean_dem - mean_rep| / scale_range
    - mean_dem, mean_rep: raw means
    - n_dem, n_rep: sample sizes
    - direction: 'Dem higher' or 'Rep higher'
    """
    # Filter valid responses (>= 1 excludes typical GSS missing codes like 0, 8, 9, -1)
    valid = df[[partyid_col, question_col]].dropna()
    valid = valid[valid[question_col] >= 1]
    
    # For most GSS opinion vars, valid responses are 1-4 or 1-3
    # Exclude high values that typically mean DK/NA (8, 9, etc.)
    max_valid = valid[question_col].quantile(0.95)  # Use 95th percentile as heuristic
    if max_valid <= 7:
        valid = valid[valid[question_col] <= 7]
    
    # Create party groups
    valid = create_party_groups(valid, partyid_col)
    valid = valid[valid['party_group'].isin(['Democrat', 'Republican'])]
    
    # Calculate means by party
    grouped = valid.groupby('party_group')[question_col]
    means = grouped.mean()
    counts = grouped.count()
    
    # Check minimum sample sizes
    if 'Democrat' not in means.index or 'Republican' not in means.index:
        return None
    if counts['Democrat'] < min_responses or counts['Republican'] < min_responses:
        return None
    
    mean_dem = means['Democrat']
    mean_rep = means['Republican']
    
    # Calculate scale range from valid data
    scale_min = valid[question_col].min()
    scale_max = valid[question_col].max()
    scale_range = scale_max - scale_min
    
    if scale_range == 0:
        return None
    
    # Normalized polarization (0-1 scale)
    polarization = abs(mean_dem - mean_rep) / scale_range
    
    return {
        'polarization': polarization,
        'mean_dem': mean_dem,
        'mean_rep': mean_rep,
        'n_dem': counts['Democrat'],
        'n_rep': counts['Republican'],
        'n_total': counts['Democrat'] + counts['Republican'],
        'scale_range': scale_range,
        'direction': 'Dem higher' if mean_dem > mean_rep else 'Rep higher'
    }

print("Polarization functions defined.")

Polarization functions defined.


In [34]:
# Create reverse mappings: variable -> topic for each category

def create_var_to_topic_mapping(topic_dict):
    """Create a mapping from variable name to topic name."""
    var_to_topic = {}
    for topic, variables in topic_dict.items():
        for var in variables:
            # Handle 'y' suffix variants - map them to base variable
            base_var = var.rstrip('y') if var.endswith('y') and not var.endswith('ay') else var
            var_to_topic[var] = topic
            var_to_topic[base_var] = topic
    return var_to_topic

PUBLIC_VAR_TO_TOPIC = create_var_to_topic_mapping(PUBLIC_ISSUES_TOPICS)
PRIVATE_VAR_TO_TOPIC = create_var_to_topic_mapping(PRIVATE_LIFE_TOPICS)

# Get all unique variables for each category
public_vars = set()
for vars_list in PUBLIC_ISSUES_TOPICS.values():
    for v in vars_list:
        # Add base variable (without 'y' suffix) to the set
        base_v = v.rstrip('y') if v.endswith('y') and not v.endswith('ay') else v
        public_vars.add(base_v)

private_vars = set()
for vars_list in PRIVATE_LIFE_TOPICS.values():
    for v in vars_list:
        base_v = v.rstrip('y') if v.endswith('y') and not v.endswith('ay') else v
        private_vars.add(base_v)

print(f"Public Issues: {len(PUBLIC_ISSUES_TOPICS)} topics, {len(public_vars)} unique variables")
print(f"Private Life: {len(PRIVATE_LIFE_TOPICS)} topics, {len(private_vars)} unique variables")

Public Issues: 60 topics, 155 unique variables
Private Life: 48 topics, 112 unique variables


In [35]:
# Calculate polarization for PUBLIC ISSUES

print("Calculating polarization for PUBLIC ISSUES...")
print("=" * 50)

public_results = []
missing_vars = []

for var in public_vars:
    if var not in df.columns:
        missing_vars.append(var)
        continue
    
    pol_result = calculate_polarization(df, var)
    if pol_result is not None:
        public_results.append({
            'variable': var,
            'area': PUBLIC_VAR_TO_TOPIC.get(var, 'Unknown'),
            **pol_result
        })

# Create results DataFrame
public_polarization_df = pd.DataFrame(public_results)
public_polarization_df = public_polarization_df.sort_values('polarization', ascending=False)

print(f"\nCalculated polarization for {len(public_polarization_df)} public issue variables")
print(f"Missing from data: {len(missing_vars)} variables")
if missing_vars:
    print(f"  Missing: {missing_vars[:10]}{'...' if len(missing_vars) > 10 else ''}")

print(f"\nTop 10 most polarizing PUBLIC ISSUES:")
print(public_polarization_df[['variable', 'area', 'polarization', 'direction', 'n_total']].head(10).to_string(index=False))

Calculating polarization for PUBLIC ISSUES...

Calculated polarization for 134 public issue variables
Missing from data: 20 variables
  Missing: ['scibnfts', 'natenrg', 'intfarm', 'watergen', 'scientod', 'conarm', 'tempgen', 'genegen', 'intmil', 'natcit']...

Top 10 most polarizing PUBLIC ISSUES:
variable                                  area  polarization  direction  n_total
 racdif1     Race: Explanations for Inequality      0.562443 Rep higher     3770
 abhelp2 Abortion: Willingness to Help/Support      0.472126 Rep higher      913
abnomore  Abortion: Circumstances for Legality      0.470704 Rep higher     3191
  abpoor  Abortion: Circumstances for Legality      0.446558 Rep higher     3212
absingle  Abortion: Circumstances for Legality      0.443313 Rep higher     3205
 abhelp3 Abortion: Willingness to Help/Support      0.439301 Rep higher      918
 natrace         Race: Government Intervention      0.418868 Rep higher     8134
 helpblk         Race: Government Intervention      0.

In [36]:
# Calculate polarization for PRIVATE LIFE

print("Calculating polarization for PRIVATE LIFE...")
print("=" * 50)

private_results = []
missing_vars_private = []

for var in private_vars:
    if var not in df.columns:
        missing_vars_private.append(var)
        continue
    
    pol_result = calculate_polarization(df, var)
    if pol_result is not None:
        private_results.append({
            'variable': var,
            'area': PRIVATE_VAR_TO_TOPIC.get(var, 'Unknown'),
            **pol_result
        })

# Create results DataFrame
private_polarization_df = pd.DataFrame(private_results)
private_polarization_df = private_polarization_df.sort_values('polarization', ascending=False)

print(f"\nCalculated polarization for {len(private_polarization_df)} private life variables")
print(f"Missing from data: {len(missing_vars_private)} variables")
if missing_vars_private:
    print(f"  Missing: {missing_vars_private[:10]}{'...' if len(missing_vars_private) > 10 else ''}")

print(f"\nTop 10 most polarizing PRIVATE LIFE topics:")
print(private_polarization_df[['variable', 'area', 'polarization', 'direction', 'n_total']].head(10).to_string(index=False))

Calculating polarization for PRIVATE LIFE...

Calculated polarization for 88 private life variables
Missing from data: 24 variables
  Missing: ['xmarsex1', 'helpfrds', 'astrosci', 'viszoo', 'hapunhap', 'helpfulv', 'nihilism', 'godmeans', 'makefrnd', 'trustsci']...

Top 10 most polarizing PRIVATE LIFE topics:
variable                              area  polarization  direction  n_total
 homosex    Sexual Morality: Homosexuality      0.324925 Dem higher     5470
religimp   Religious Identity & Commitment      0.227485 Dem higher     2794
savesoul         Religious Social Function      0.223262 Dem higher     6029
premarsx       Sexual Morality: Premarital      0.195853 Dem higher     5532
relidimp   Religious Identity & Commitment      0.193470 Dem higher     2722
spanking             Parenting: Discipline      0.192885 Dem higher     5564
     god Religious Belief: God & Afterlife      0.188705 Rep higher     5568
 teensex       Sexual Morality: Premarital      0.186717 Dem higher     55

In [37]:
# Save results to CSV files
# Column order: variable, area, polarization, mean_dem, mean_rep, n_dem, n_rep, n_total, scale_range, direction

column_order = ['variable', 'area', 'polarization', 'mean_dem', 'mean_rep', 'n_dem', 'n_rep', 'n_total', 'scale_range', 'direction']

# Save public issues polarization
public_polarization_df = public_polarization_df[column_order]
public_polarization_df.to_csv('../../data/polarization/public_issues_polarization.csv', index=False)
print(f"Saved public_issues_polarization.csv ({len(public_polarization_df)} rows)")

# Save private life polarization
private_polarization_df = private_polarization_df[column_order]
private_polarization_df.to_csv('../../data/polarization/private_life_polarization.csv', index=False)
print(f"Saved private_life_polarization.csv ({len(private_polarization_df)} rows)")

Saved public_issues_polarization.csv (134 rows)
Saved private_life_polarization.csv (88 rows)


In [38]:
# Summary statistics

print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)

print("\n--- PUBLIC ISSUES ---")
print(f"Total questions analyzed: {len(public_polarization_df)}")
print(f"Total topics: {public_polarization_df['area'].nunique()}")
print(f"Mean polarization: {public_polarization_df['polarization'].mean():.3f}")
print(f"Max polarization: {public_polarization_df['polarization'].max():.3f} ({public_polarization_df.iloc[0]['variable']})")
print(f"Min polarization: {public_polarization_df['polarization'].min():.3f} ({public_polarization_df.iloc[-1]['variable']})")

print("\n--- PRIVATE LIFE ---")
print(f"Total questions analyzed: {len(private_polarization_df)}")
print(f"Total topics: {private_polarization_df['area'].nunique()}")
print(f"Mean polarization: {private_polarization_df['polarization'].mean():.3f}")
print(f"Max polarization: {private_polarization_df['polarization'].max():.3f} ({private_polarization_df.iloc[0]['variable']})")
print(f"Min polarization: {private_polarization_df['polarization'].min():.3f} ({private_polarization_df.iloc[-1]['variable']})")

print("\n--- COMPARISON ---")
print(f"Public Issues mean polarization: {public_polarization_df['polarization'].mean():.3f}")
print(f"Private Life mean polarization: {private_polarization_df['polarization'].mean():.3f}")


SUMMARY

--- PUBLIC ISSUES ---
Total questions analyzed: 134
Total topics: 58
Mean polarization: 0.182
Max polarization: 0.562 (racdif1)
Min polarization: 0.006 (dangroth)

--- PRIVATE LIFE ---
Total questions analyzed: 88
Total topics: 44
Mean polarization: 0.068
Max polarization: 0.325 (homosex)
Min polarization: 0.002 (emailhr)

--- COMPARISON ---
Public Issues mean polarization: 0.182
Private Life mean polarization: 0.068


In [39]:
# Polarization by topic for PUBLIC ISSUES

print("PUBLIC ISSUES - Polarization by Topic:")
print("=" * 70)

public_topic_stats = public_polarization_df.groupby('area').agg({
    'polarization': ['mean', 'std', 'count'],
    'n_total': 'mean'
}).round(3)

public_topic_stats.columns = ['mean_polarization', 'std_polarization', 'n_questions', 'avg_sample_size']
public_topic_stats = public_topic_stats.sort_values('mean_polarization', ascending=False)
print(public_topic_stats.to_string())

PUBLIC ISSUES - Polarization by Topic:
                                                mean_polarization  std_polarization  n_questions  avg_sample_size
area                                                                                                             
Race: Government Intervention                               0.417             0.002            2         6818.500
Political Ideology                                          0.359               NaN            1         8325.000
Abortion: Willingness to Help/Support                       0.349             0.143            4          918.000
Spending: Environment & Energy                              0.342               NaN            1         8356.000
Military Spending                                           0.333               NaN            1         8310.000
Government: Role & Size                                     0.324               NaN            1         5493.000
Abortion: Circumstances for Legality             

In [40]:
print("PRIVATE LIFE - Polarization by Question:")
print("=" * 100)

# Sort by polarization descending and display all columns
public_polarization_sorted = public_polarization_df.sort_values('polarization', ascending=False)
print(public_polarization_sorted[['variable', 'area', 'polarization', 'mean_dem', 'mean_rep', 'direction', 'n_total']].to_string(index=False))

print(f"\nTotal questions: {len(public_polarization_sorted)}")

PRIVATE LIFE - Polarization by Question:
 variable                                           area  polarization  mean_dem  mean_rep  direction  n_total
  racdif1              Race: Explanations for Inequality      0.562443  1.202561  1.765003 Rep higher     3770
  abhelp2          Abortion: Willingness to Help/Support      0.472126  1.328413  1.800539 Rep higher      913
 abnomore           Abortion: Circumstances for Legality      0.470704  1.202303  1.673007 Rep higher     3191
   abpoor           Abortion: Circumstances for Legality      0.446558  1.220596  1.667154 Rep higher     3212
 absingle           Abortion: Circumstances for Legality      0.443313  1.235390  1.678703 Rep higher     3205
  abhelp3          Abortion: Willingness to Help/Support      0.439301  1.169742  1.609043 Rep higher      918
  natrace                  Race: Government Intervention      0.418868  1.311275  2.149012 Rep higher     8134
  helpblk                  Race: Government Intervention      0.415418 

In [41]:
# Polarization by topic for PRIVATE LIFE

print("PRIVATE LIFE - Polarization by Topic:")
print("=" * 70)

private_topic_stats = private_polarization_df.groupby('area').agg({
    'polarization': ['mean', 'std', 'count'],
    'n_total': 'mean'
}).round(3)

private_topic_stats.columns = ['mean_polarization', 'std_polarization', 'n_questions', 'avg_sample_size']
private_topic_stats = private_topic_stats.sort_values('mean_polarization', ascending=False)
print(private_topic_stats.to_string())

PRIVATE LIFE - Polarization by Topic:
                                         mean_polarization  std_polarization  n_questions  avg_sample_size
area                                                                                                      
Sexual Morality: Homosexuality                       0.325               NaN            1         5470.000
Religious Social Function                            0.223               NaN            1         6029.000
Parenting: Discipline                                0.193               NaN            1         5564.000
Sexual Morality: Premarital                          0.191             0.006            2         5536.500
Religious Belief: God & Afterlife                    0.148             0.042            4         3994.000
Leisure: Outdoor & Cultural                          0.146               NaN            1         6641.000
Religious Practice: Attendance & Prayer              0.141             0.036            2         7142.000

In [42]:
print("PRIVATE LIFE - Polarization by Question:")
print("=" * 100)

# Sort by polarization descending and display all columns
private_polarization_sorted = private_polarization_df.sort_values('polarization', ascending=False)
print(private_polarization_sorted[['variable', 'area', 'polarization', 'mean_dem', 'mean_rep', 'direction', 'n_total']].to_string(index=False))

print(f"\nTotal questions: {len(private_polarization_sorted)}")


PRIVATE LIFE - Polarization by Question:
variable                                    area  polarization  mean_dem  mean_rep  direction  n_total
 homosex          Sexual Morality: Homosexuality      0.324925  3.358102  2.383326 Dem higher     5470
religimp         Religious Identity & Commitment      0.227485  2.637376  1.954922 Dem higher     2794
savesoul               Religious Social Function      0.223262  1.712849  1.489588 Dem higher     6029
premarsx             Sexual Morality: Premarital      0.195853  3.558323  2.970763 Dem higher     5532
relidimp         Religious Identity & Commitment      0.193470  3.088592  2.314711 Dem higher     2722
spanking                   Parenting: Discipline      0.192885  2.705606  2.126951 Dem higher     5564
     god       Religious Belief: God & Afterlife      0.188705  4.288050  5.231575 Rep higher     5568
 teensex             Sexual Morality: Premarital      0.186717  2.277690  1.717538 Dem higher     5541
  reborn       Religious Belief: